# LeakAudit - 06 - Recording-level leakage on D3 (E5)

**Cutting each 30 s hydrophone clip into one-second segments, then splitting at random, scatters segments of the same recording across train and test.**

- This reproduces failure mode F1 (Kapoor & Narayanan 2023, L3.2) in a site-free setting, so the effect is isolated from site memorisation.
- Three protocols: segment-random -> recording-independent -> condition-independent.

**Reading the three protocols correctly.** Only the first step isolates
recording leakage:

| comparison | what it measures |
|---|---|
| S1 -> S2 | **recording leakage alone** - same conditions, segments no longer shared |
| S2 -> S3 | **an additional domain shift** to unseen experimental conditions |

The full S1 -> S3 drop therefore must not be attributed to segment sharing.
Note also that S3 yields fewer usable repeats than S1 and S2, because some
condition-wise folds contain a single class; the printed `n` per protocol
records this.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GroupShuffleSplit

M3 = io_utils.scan_d3()
MODALITY = "hydrophone"
WIN_SEC = 1.0
# 30 s clips at 8 kHz give exactly 30 one-second segments per recording.
MAX_SEG = 30

## Build one-second segments (cached)

In [3]:
def build_segments(win_sec=WIN_SEC, modality=MODALITY, max_seg=MAX_SEG):
    fp = cfg.CACHE / f"d3_seg_{modality}_{win_sec}.npz"
    if fp.exists():
        z = np.load(fp, allow_pickle=True)
        return z["X"], pd.DataFrame(z["meta"], columns=z["cols"])
    sub = M3[(~M3.is_bg) & (M3.modality == modality) & M3.label.notna()].reset_index(drop=True)
    Xs, rows = [], []
    for i, r in tqdm(sub.iterrows(), total=len(sub), desc="segments"):
        x, sr = io_utils.load_d3_signal(r.to_dict())
        xb = features.bandpass(x, sr)
        n = int(win_sec * sr); k = min(max_seg, len(xb) // n)
        for j in range(k):
            Xs.append(features.summary_features(xb[j*n:(j+1)*n], sr))
            rows.append(dict(rec_id=i, seg=j, label=r["label"],
                             topology=r["topology"], leaktype=r["leaktype"]))
    X = np.vstack(Xs); meta = pd.DataFrame(rows)
    np.savez_compressed(fp, X=X, meta=meta.to_numpy(), cols=meta.columns.to_numpy())
    return X, meta

Xseg, Mseg = build_segments()
print(f"segments: {Xseg.shape}  from {Mseg.rec_id.nunique()} recordings")
print(Mseg.label.value_counts().to_string())

segments: 100%|██████████| 120/120 [00:08<00:00, 14.40it/s]


segments: (3600, 199)  from 120 recordings
label
leak      2880
noleak     720


## E5 - three split protocols on the same segments

In [4]:
yseg = Mseg.label.to_numpy()
rec  = Mseg.rec_id.to_numpy()
cond = (Mseg.topology.astype(str) + "_" + Mseg.leaktype.astype(str)).to_numpy()

rows = []
for seed in range(cfg.N_SEEDS):
    a, b = train_test_split(np.arange(len(yseg)), test_size=.3,
                            random_state=seed, stratify=yseg)
    rows.append(dict(protocol="S1_segment_random", seed=seed,
                     **evaluation.evaluate(Xseg[a], yseg[a], Xseg[b], yseg[b], "gbdt", seed)))
    a, b = next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                .split(np.arange(len(yseg)), yseg, groups=rec))
    if len(np.unique(yseg[a])) > 1 and len(np.unique(yseg[b])) > 1:
        rows.append(dict(protocol="S2_recording_indep", seed=seed,
                         **evaluation.evaluate(Xseg[a], yseg[a], Xseg[b], yseg[b], "gbdt", seed)))
    a, b = next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                .split(np.arange(len(yseg)), yseg, groups=cond))
    if len(np.unique(yseg[a])) > 1 and len(np.unique(yseg[b])) > 1:
        rows.append(dict(protocol="S3_condition_indep", seed=seed,
                         **evaluation.evaluate(Xseg[a], yseg[a], Xseg[b], yseg[b], "gbdt", seed)))

e5 = pd.DataFrame(rows)
s5 = e5.groupby("protocol").agg(acc=("acc", "mean"), sd=("acc", "std"),
                                bacc=("bacc", "mean"), base=("base", "mean"),
                                n=("acc", "size")).reset_index()
print(s5.round(4).to_string(index=False))
e5.to_csv(cfg.RESULTS / "E5_recording_leakage_raw.csv", index=False)
s5.to_csv(cfg.RESULTS / "E5_recording_leakage_summary.csv", index=False)
# Decompose the drop rather than quoting the end-to-end difference, which
# would credit recording leakage with an effect that is partly domain shift.
g = s5.set_index("protocol").acc
if {"S1_segment_random", "S2_recording_indep"} <= set(g.index):
    d_leak = g["S1_segment_random"] - g["S2_recording_indep"]
    print(f"\nrecording leakage alone (S1 -> S2)      : {d_leak:+.4f}")
if {"S2_recording_indep", "S3_condition_indep"} <= set(g.index):
    d_shift = g["S2_recording_indep"] - g["S3_condition_indep"]
    print(f"additional unseen-condition shift (S2->S3): {d_shift:+.4f}")
if {"S1_segment_random", "S3_condition_indep"} <= set(g.index):
    print(f"end-to-end (S1 -> S3, do NOT attribute to leakage alone): "
          f"{g['S1_segment_random'] - g['S3_condition_indep']:+.4f}")
print("\nrepeats per protocol:")
print(s5[["protocol", "n"]].to_string(index=False))

          protocol    acc     sd   bacc   base  n
 S1_segment_random 0.9881 0.0023 0.9757 0.8000 10
S2_recording_indep 0.9052 0.0467 0.7952 0.8056 10
S3_condition_indep 0.5950 0.1102 0.4489 0.6667  5

recording leakage alone (S1 -> S2)      : +0.0829
additional unseen-condition shift (S2->S3): +0.3102
end-to-end (S1 -> S3, do NOT attribute to leakage alone): +0.3931

repeats per protocol:
          protocol  n
 S1_segment_random 10
S2_recording_indep 10
S3_condition_indep  5
